# Week 14 — Engineering Simulation and Computational Modeling

**ESC 113: Computer Aided Analysis for Engineering**

This notebook introduces engineering simulation as the process of converting physical laws and engineering assumptions into computational models that evolve over time.

## Table of Contents

1. Engineering Simulation Concepts
2. Modeling Assumptions
3. Time-Stepping
4. Mechanical Simulation
5. Thermal Simulation
6. RC Circuit Simulation
7. Tank Simulation
8. Parameter Studies
9. Numerical Stability
10. Verification and Validation
11. Guided Exercises
12. Review Questions

## 1. What Is Engineering Simulation?

Engineering simulation uses mathematical models and numerical computation to study how a system behaves.

Typical workflow:

**physical system → assumptions → equations → numerical algorithm → simulation → interpretation**

Examples:
- temperature evolution;
- structural response;
- electrical circuits;
- fluid systems;
- vehicle motion;
- battery discharge;
- process control.

## 2. Inputs, Parameters, States, and Outputs

Example: cooling object

- input: ambient temperature
- parameter: cooling constant k
- state: object temperature T
- output: temperature as a function of time

A simulation model should clearly distinguish these roles.

## 3. Euler Time-Stepping

For a differential equation:

dy/dt = f(y,t)

the forward-Euler approximation is:

y_next = y_current + Δt × f(y_current,t)

This is simple and useful for learning, but time-step size can strongly affect accuracy and stability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dt = 0.1
t = np.arange(0, 10 + dt, dt)

y = np.zeros_like(t)
y[0] = 1.0

for i in range(len(t)-1):
    dydt = -0.5 * y[i]
    y[i+1] = y[i] + dt * dydt

plt.plot(t, y)
plt.xlabel("Time")
plt.ylabel("y")
plt.title("Simple Euler Simulation")
plt.grid(True)
plt.show()

## 4. Mechanical Simulation — Falling Object with Linear Drag

Model:

m dv/dt = mg − cv

Therefore:

dv/dt = g − (c/m)v

In [ ]:
m = 80.0
c = 12.0
g = 9.81

dt = 0.05
t = np.arange(0, 30 + dt, dt)

v = np.zeros_like(t)

for i in range(len(t)-1):
    dvdt = g - (c/m) * v[i]
    v[i+1] = v[i] + dt * dvdt

plt.plot(t, v)
plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.title("Falling Object with Linear Drag")
plt.grid(True)
plt.show()

print("Simulated final velocity:", v[-1])
print("Analytical terminal velocity:", m*g/c)

## 5. Parameter Study — Drag Coefficient

In [ ]:
drag_values = [5, 10, 15, 20]

plt.figure()

for c in drag_values:
    v = np.zeros_like(t)

    for i in range(len(t)-1):
        dvdt = g - (c/m) * v[i]
        v[i+1] = v[i] + dt * dvdt

    plt.plot(t, v, label=f"c={c}")

plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.title("Effect of Drag Coefficient")
plt.legend()
plt.grid(True)
plt.show()

## 6. Thermal Simulation — Newton Cooling

Model:

dT/dt = −k(T − T_ambient)

In [ ]:
T0 = 95.0
T_ambient = 22.0
k = 0.08

dt = 0.1
t = np.arange(0, 60 + dt, dt)

T = np.zeros_like(t)
T[0] = T0

for i in range(len(t)-1):
    dTdt = -k * (T[i] - T_ambient)
    T[i+1] = T[i] + dt * dTdt

plt.plot(t, T)
plt.xlabel("Time (min)")
plt.ylabel("Temperature (°C)")
plt.title("Cooling Simulation")
plt.grid(True)
plt.show()

## 7. RC Circuit Simulation

Charging capacitor:

dV/dt = (V_source − V)/(RC)

In [ ]:
Vs = 12.0
R = 1000.0
C = 0.001

dt = 0.01
t = np.arange(0, 6 + dt, dt)

V = np.zeros_like(t)

for i in range(len(t)-1):
    dVdt = (Vs - V[i]) / (R*C)
    V[i+1] = V[i] + dt * dVdt

plt.plot(t, V)
plt.xlabel("Time (s)")
plt.ylabel("Capacitor Voltage (V)")
plt.title("RC Charging Simulation")
plt.grid(True)
plt.show()

## 8. Compare with Analytical RC Solution

In [ ]:
V_exact = Vs * (1 - np.exp(-t/(R*C)))

plt.plot(t, V, label="Euler")
plt.plot(t, V_exact, "--", label="Analytical")
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("Numerical vs Analytical RC Response")
plt.legend()
plt.grid(True)
plt.show()

## 9. Tank Filling/Draining Simulation

Balance:

dV/dt = Q_in − Q_out

In [ ]:
Q_in = 18.0
Q_out = 12.0
capacity = 500.0

dt = 1.0
t = np.arange(0, 120 + dt, dt)

volume = np.zeros_like(t)
volume[0] = 100.0

for i in range(len(t)-1):
    dVdt = Q_in - Q_out
    volume[i+1] = volume[i] + dt * dVdt

    if volume[i+1] >= capacity:
        volume[i+1:] = capacity
        print("Capacity reached at t =", t[i+1])
        break

plt.plot(t, volume)
plt.xlabel("Time (min)")
plt.ylabel("Volume (L)")
plt.title("Tank Volume Simulation")
plt.grid(True)
plt.show()

## 10. Numerical Stability and Time Step

A smaller time step generally improves accuracy but increases computation.

A time step that is too large can produce:
- inaccurate solutions;
- oscillations;
- divergence;
- physically impossible values.

Always perform a time-step sensitivity check.

In [ ]:
def cooling_simulation(dt):
    t = np.arange(0, 60 + dt, dt)
    T = np.zeros_like(t)
    T[0] = 95.0

    for i in range(len(t)-1):
        T[i+1] = T[i] + dt * (-0.08*(T[i]-22))

    return t, T

for dt in [0.05, 0.5, 2.0, 10.0]:
    t_sim, T_sim = cooling_simulation(dt)
    plt.plot(t_sim, T_sim, label=f"dt={dt}")

plt.xlabel("Time")
plt.ylabel("Temperature")
plt.title("Effect of Time Step")
plt.legend()
plt.grid(True)
plt.show()

## 11. Verification vs Validation

**Verification:** Did we solve the equations correctly?

Examples:
- compare against an analytical solution;
- reduce time step and check convergence;
- test limiting cases.

**Validation:** Do the equations represent the real system adequately?

Examples:
- compare simulation with experiments;
- compare with field measurements;
- evaluate prediction error under real operating conditions.

## 12. Guided Exercises

1. Modify the cooling model for three ambient temperatures.
2. Modify the RC model for three resistance values.
3. Add an overflow warning to the tank model.
4. Calculate the time required for the falling object to reach 95% of terminal velocity.
5. Compare Euler results using two different time steps.
6. Create one new engineering simulation using the same time-stepping pattern.

## 13. Review Questions

1. What is a state variable?
2. What is the purpose of a time step?
3. What is parameter sensitivity?
4. Why should numerical results be compared with analytical results when possible?
5. What is the difference between verification and validation?
6. Why can a large time step be dangerous?
7. Why must model assumptions be documented?
8. Why should simulation results not automatically be treated as real-world truth?